# 01 · Chạy inference theo NHÓM model (mọi variant)

Chạy **00_setup trước**. Mỗi NHÓM (family) chạy trong **1 session/env riêng** (deps xung đột).
Notebook tự chạy TẤT CẢ variant của nhóm (vd zipformer chunk 16/32/64) để dựng
đường cong đánh đổi độ chính xác ↔ tốc độ. Resume-được.


In [ ]:
%cd /content/benchmark-for-asr-model
!git pull -q
import glob, os
FAMILY = 'zipformer'   # zipformer | gipformer | chunkformer | nemotron
DATASETS = ['auto_autoele', 'sol_ups']
CONFIGS = sorted(glob.glob(f'configs/models/{FAMILY}*.yaml'))
print('variant sẽ chạy:'); [print(' ', c) for c in CONFIGS]

### Cài deps của nhóm
> **nemotron**: bỏ comment dòng apt trước khi cài.


In [ ]:
# !apt-get -qq install -y libsndfile1 ffmpeg && pip install -q Cython packaging  # chỉ nemotron
!pip install -q -r requirements/{FAMILY}.txt

### (Chỉ zipformer) sinh tokens.txt từ bpe.model
Các config zipformer đã trỏ `tokens: models/zipformer_tokens.txt`.


In [ ]:
if FAMILY == 'zipformer':
    from huggingface_hub import hf_hub_download
    import sentencepiece as spm
    os.makedirs('models', exist_ok=True)
    bpe = hf_hub_download('hynt/Zipformer-30M-RNNT-Streaming-6000h', 'bpe.model')
    sp = spm.SentencePieceProcessor(); sp.load(bpe)
    with open('models/zipformer_tokens.txt','w') as f:
        for i in range(sp.get_piece_size()):
            f.write(f'{sp.id_to_piece(i)} {i}\n')
    print('tokens.txt:', sp.get_piece_size(), 'dòng')

### Smoke-test 5 utterance (variant đầu tiên)


In [ ]:
cfg0 = CONFIGS[0]
!PYTHONPATH=src python -m benchmark.runners.run_inference \
  --model-config {cfg0} --manifest data/manifests/auto_autoele.jsonl \
  --out /tmp/smoke.jsonl --limit 5 --no-resume --warmup 0
print(open('/tmp/smoke.jsonl').read())

### Chạy full mọi variant × mọi dataset (resume-được, lưu ra Drive)


In [ ]:
for cfg in CONFIGS:
    stem = os.path.splitext(os.path.basename(cfg))[0]
    for ds in DATASETS:
        print(f'=== {stem} / {ds} ===')
        !PYTHONPATH=src python -m benchmark.runners.run_inference \
            --model-config {cfg} --manifest data/manifests/{ds}.jsonl \
            --out results/hypotheses/{stem}/{ds}.jsonl --warmup 3